# Validación Visual y Reproducible de la Fórmula Explícita de Weil

Este notebook demuestra, visualiza y explica paso a paso la validación numérica de la fórmula explícita de Weil para la hipótesis de Riemann, usando el código y los datos de este repositorio.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        candidate
        for candidate in candidates
        if (candidate / 'validate_explicit_formula.py').exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError('No se pudo localizar la raíz del repositorio desde el notebook.')

os.chdir(repo_root)
print(f'Usando raíz del repositorio: {repo_root}')


## Ejecución del validador y carga de resultados


In [ ]:
command = [
    sys.executable,
    str(repo_root / 'validate_explicit_formula.py'),
    '--use_weil_formula',
    '--max_primes',
    '2000',
    '--max_zeros',
    '200',
    '--prime_powers',
    '5',
    '--integration_t',
    '10',
    '--precision_dps',
    '40',
    '--test_functions',
    'gaussian',
]
subprocess.run(command, check=True)

results_path = repo_root / 'data' / 'validation_results.csv'
results = pd.read_csv(results_path)
display(results)


## Visualización de ceros simulados vs. reales


In [ ]:
zeros_path = repo_root / 'zeros' / 'zeros_t1e8.txt'
with zeros_path.open() as handle:
    zeros_reales = [float(line.strip()) for line in handle][:200]

from validate_explicit_formula import simulate_delta_s

_, ceros_simulados, _ = simulate_delta_s(200, precision=40)

plt.figure(figsize=(10, 5))
plt.plot(zeros_reales, label='Ceros reales (Odlyzko)', marker='o', linestyle='None', alpha=0.5)
plt.plot(ceros_simulados, label='Ceros simulados (ΔS adélico)', marker='x', linestyle='None', alpha=0.7)
plt.xlabel('Índice')
plt.ylabel('Parte imaginaria')
plt.title('Comparación de ceros no triviales')
plt.legend()
plt.grid(True)
plt.show()


## Comparación numérica de ambos lados de la fórmula


In [ ]:
def metric(name: str) -> float:
    return float(results.loc[results['parameter'] == name, 'value'].iloc[0])

left = metric('left_side')
right = metric('right_side')
error = metric('absolute_error')
rel_error = metric('relative_error')

plt.figure(figsize=(7, 4))
plt.bar(['Lado ceros+arch'], [abs(left)], color='royalblue')
plt.bar(['Lado primos+arch'], [abs(right)], color='darkorange')
plt.title('Magnitud de ambos lados de la fórmula explícita')
plt.ylabel('Valor absoluto')
plt.show()

print(f'Error absoluto: {error}')
print(f'Error relativo: {rel_error}')


## Interpretación y conclusiones

La diferencia numérica entre ambos lados refleja la precisión de la simulación y la calidad de la aproximación espectral y aritmética. Un error relativo pequeño valida la consistencia numérica de la fórmula explícita bajo el enfoque adélico.

> "La matemática es la música de la razón."
